# Car Price Prediction

## Objective

**What is the problem?**
Predicting the selling price of used cars using features such as manufacturing year, fuel type, seller type, and transmission.

**Why is it useful?**
It helps buyers and sellers determine fair market values and uncovers the factors that most heavily influence vehicle depreciation.

**Expected outcome.**
A trained regression model that yields reliable price predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

## Dataset Loading

In [ ]:
dataset_path = '../dataset/car_data.csv'
if not os.path.exists(dataset_path):
    print('Please place the CarDekho dataset as car_data.csv in the dataset/ folder.')
else:
    df = pd.read_csv(dataset_path)
    display(df.head())
    display(df.tail())
    display(df.sample(5))
    print('Shape:', df.shape)
    print('Columns:', df.columns)
    df.info()
    display(df.describe())
    print('Data Types:\n', df.dtypes)

## Data Cleaning
Handling missing values, duplicate rows, and category inconsistencies.

In [ ]:
if os.path.exists(dataset_path):
    print('Null values:\n', df.isnull().sum())
    print('Duplicates:', df.duplicated().sum())
    df.drop_duplicates(inplace=True)
    print('Shape after dropping duplicates:', df.shape)

## Feature Engineering
Creating a new `Car_Age` feature which is much more informative for models than the raw `Year` of manufacturing. We also extract the brand if applicable, but for simplicity here we rely on the core features.

In [ ]:
if os.path.exists(dataset_path) and 'Year' in df.columns:
    current_year = 2024
    df['Car_Age'] = current_year - df['Year']
    df.drop(['Year'], axis=1, inplace=True)
    if 'Car_Name' in df.columns:
        df.drop(['Car_Name'], axis=1, inplace=True) # Car_Name is highly cardinal
    display(df.head())

### Why it is done
Models prefer numerical continuous or categorical representations. `Car_Age` represents depreciation perfectly. Dropping `Car_Name` reduces extreme sparsity from one-hot encoding hundreds of unique names.

## Exploratory Data Analysis

In [ ]:
if os.path.exists(dataset_path) and 'Selling_Price' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Selling_Price'], kde=True, color='blue')
    plt.title('Price Distribution')
    plt.savefig('../images/price_dist.png')
    plt.show()

### Observations
- The target variable `Selling_Price` is right-skewed. Most cars are sold at lower price points.

In [ ]:
if os.path.exists(dataset_path) and 'Fuel_Type' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='Fuel_Type', y='Selling_Price')
    plt.title('Selling Price vs Fuel Type')
    plt.savefig('../images/fuel_box.png')
    plt.show()

### Observations
- Diesel cars generally have a higher median selling price compared to Petrol or CNG cars.

In [ ]:
if os.path.exists(dataset_path) and 'Transmission' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='Transmission', y='Selling_Price')
    plt.title('Selling Price vs Transmission')
    plt.savefig('../images/transmission_box.png')
    plt.show()

### Observations
- Automatic transmission cars fetch significantly higher prices than manual ones.

In [ ]:
if os.path.exists(dataset_path) and 'Car_Age' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='Car_Age', y='Selling_Price')
    plt.title('Selling Price vs Car Age')
    plt.savefig('../images/price_vs_age.png')
    plt.show()

### Observations
- Clear negative correlation; as the car ages, its selling price drops.

In [ ]:
if os.path.exists(dataset_path):
    plt.figure(figsize=(10, 8))
    corr = df.corr(numeric_only=True)
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Heatmap')
    plt.savefig('../images/correlation_heatmap.png')
    plt.show()

### Observations
- `Present_Price` is highly positively correlated with `Selling_Price`.
- `Car_Age` has a strong negative correlation with `Selling_Price`.

## Encoding Categorical Variables
We apply One Hot Encoding to convert categorical text data into numerical binary data suitable for regression.

In [ ]:
if os.path.exists(dataset_path):
    df = pd.get_dummies(df, drop_first=True)
    display(df.head())

## Train Test Split

In [ ]:
if os.path.exists(dataset_path) and 'Selling_Price' in df.columns:
    X = df.drop('Selling_Price', axis=1)
    y = df['Selling_Price']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Training
We train Linear Regression (baseline), Random Forest, and Gradient Boosting Regressor.

In [ ]:
if os.path.exists(dataset_path) and 'Selling_Price' in df.columns:
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(random_state=42)
    }
    results = {}

## Model Evaluation

In [ ]:
if os.path.exists(dataset_path) and 'Selling_Price' in df.columns:
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        results[name] = r2
        print(f'--- {name} ---')
        print(f'MAE: {mae:.2f}')
        print(f'RMSE: {rmse:.2f}')
        print(f'R2 Score: {r2:.2f}\n')

### Metric Explanations
- **MAE**: Mean Absolute Error represents the average absolute difference between actual and predicted prices.
- **RMSE**: Root Mean Squared Error penalizes larger errors more heavily.
- **R² Score**: Represents the proportion of variance in the target variable explained by the model. Closer to 1 is better.

## Feature Importance Plot

In [ ]:
if os.path.exists(dataset_path) and 'Selling_Price' in df.columns:
    best_model_name = max(results, key=results.get)
    best_model = models[best_model_name]
    
    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
        indices = np.argsort(importances)[::-1]
        plt.figure(figsize=(10, 6))
        sns.barplot(x=importances[indices], y=X.columns[indices], palette='viridis')
        plt.title('Feature Importances')
        plt.savefig('../images/feature_importance.png')
        plt.show()

## Save Best Model

In [ ]:
if os.path.exists(dataset_path) and 'Selling_Price' in df.columns:
    joblib.dump(best_model, '../models/best_car_model.pkl')
    print(f'Best model ({best_model_name}) saved to models/')

## Conclusion
- The Tree-based models (Random Forest, Gradient Boosting) substantially outperform simple Linear Regression.
- `Present_Price` and `Car_Age` are the most significant predictors.

### Future Improvements
- Deploying the model using a web framework.
- Tuning hyperparameters for Gradient Boosting using GridSearchCV.